# 🧠 Learning Rate Scheduling and Cosine Annealing

Welcome to the hands-on explanation notebook for **Learning Rate Scheduling**! In this notebook, we will:
1. Define the mathematical formulas for Linear Warmup, Step Decay, and Cosine Annealing schedulers.
2. Implement these schedulers from scratch in Python.
3. Plot and compare the scheduler curves over 100 training epochs (YOLO-style).
4. Simulate 2D Gradient Descent on $f(x,y) = x^2 + 3y^2$ comparing a **Constant Learning Rate** vs. a **Cosine Annealing Learning Rate**.
5. Plot optimization trajectories on a 2D contour map to observe how decaying the learning rate stabilizes convergence at the global minimum.
6. Connect these configurations to YOLO's default training hyperparameters.

Let's start by importing the necessary libraries.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Set seed for reproducibility
np.random.seed(42)

## 1. Implementing Schedulers from Scratch

Let's write custom scheduling functions:
-   **Step Decay:** Drops the learning rate by a multiplier (e.g. 0.5) every 20 epochs.
-   **Cosine Annealing with Warmup:** Linearly warms up the learning rate for the first 10 epochs, then decays it following a cosine curve to a minimum target.

In [ ]:
def step_decay(epoch, lr_initial=0.1, step_size=20, gamma=0.5):
    return lr_initial * (gamma ** (epoch // step_size))

def cosine_annealing_with_warmup(epoch, total_epochs=100, warmup_epochs=10, lr_max=0.1, lr_min=0.001):
    if epoch < warmup_epochs:
        # Linear Warmup
        return lr_min + (epoch / warmup_epochs) * (lr_max - lr_min)
    else:
        # Cosine Annealing
        t_cur = epoch - warmup_epochs
        t_max = total_epochs - warmup_epochs
        return lr_min + 0.5 * (lr_max - lr_min) * (1.0 + np.cos((t_cur / t_max) * np.pi))

Let's plot both schedules over 100 epochs.

In [ ]:
epochs = np.arange(100)
lr_step = [step_decay(e) for e in epochs]
lr_cosine = [cosine_annealing_with_warmup(e) for e in epochs]

plt.figure(figsize=(10, 6))
plt.plot(epochs, lr_step, color='purple', linewidth=2.5, label='Step Decay (Gamma=0.5, Step=20)')
plt.plot(epochs, lr_cosine, color='teal', linewidth=3, label='Cosine Annealing with Warmup')
plt.xlabel('Epoch')
plt.ylabel('Learning Rate (lr)')
plt.title('Learning Rate Schedules')
plt.grid(True, linestyle='--', alpha=0.5)
plt.legend()
plt.show()

## 2. Paraboloid Optimization Trajectory Comparison

Let's optimize the 2D paraboloid $f(x, y) = x^2 + 3y^2$ starting at $(8.0, 8.0)$ using:
1.  **Constant Learning Rate ($\alpha = 0.35$):** Bounces back and forth, overshooting the minimum.
2.  **Cosine Annealing Learning Rate ($\alpha_{initial} = 0.35$):** Starts fast, then slows down, settling smoothly at $(0,0)$.

In [ ]:
def cost_func(x, y):
    return x**2 + 3 * y**2

def grad_func(x, y):
    return np.array([2*x, 6*y])

def optimize_constant_lr(start_pos, lr=0.32, epochs=30):
    pos = np.array(start_pos, dtype=float)
    history = [pos.copy()]
    for _ in range(epochs):
        grad = grad_func(pos[0], pos[1])
        pos -= lr * grad
        history.append(pos.copy())
    return np.array(history)

def optimize_cosine_lr(start_pos, lr_max=0.32, lr_min=0.001, epochs=30):
    pos = np.array(start_pos, dtype=float)
    history = [pos.copy()]
    for epoch in range(epochs):
        lr = lr_min + 0.5 * (lr_max - lr_min) * (1.0 + np.cos((epoch / epochs) * np.pi))
        grad = grad_func(pos[0], pos[1])
        pos -= lr * grad
        history.append(pos.copy())
    return np.array(history)

start = [8.0, 8.0]
path_const = optimize_constant_lr(start)
path_cosine = optimize_cosine_lr(start)

Let's plot both paths on the 2D contour map.

In [ ]:
x = np.linspace(-10, 10, 100)
y = np.linspace(-10, 10, 100)
X, Y = np.meshgrid(x, y)
Z = cost_func(X, Y)

plt.figure(figsize=(10, 8))
contours = plt.contour(X, Y, Z, levels=20, cmap='viridis')
plt.clabel(contours, inline=1, fontsize=8)

plt.plot(path_const[:, 0], path_const[:, 1], color='red', marker='o', linewidth=1.5, label='Constant LR (Wiggles/Overshoots)')
plt.plot(path_cosine[:, 0], path_cosine[:, 1], color='teal', marker='s', linewidth=2.5, label='Cosine Annealing LR (Stable Convergence)')

plt.scatter(0, 0, color='blue', s=120, marker='*', zorder=5, label='Minimum')
plt.xlabel('x')
plt.ylabel('y')
plt.title('Constant Learning Rate vs. Cosine Annealing')
plt.legend()
plt.show()

Observe:
-   **Constant LR (Red):** Oscillates wildly across the $y$-axis due to the steep slope, overshooting the center.
-   **Cosine Annealing (Teal):** The early large step sizes allow it to traverse the space quickly. As it nears the center, the learning rate shrinks, dampening the oscillations and letting the model settle perfectly at the global minimum.

## 💡 Connection to YOLO and Deep Learning
*   **YOLO Hyperparameters:** YOLO defines these scheduling parameters inside its default settings:
    -   `lr0`: Initial learning rate (e.g. `0.01`).
    -   `lrf`: Final learning rate fraction (e.g. `0.01`, meaning final lr is `lr0 * lrf`).
    -   `warmup_epochs`: Duration of linear warmup phase (default `3.0` epochs).
    -   `warmup_bias_lr`: Warmup learning rate for biases (default `0.1`).
*   This ensures that the weights do not explode during early backpropagation steps and settle into stable local/global minima at the end of training.